# Conversión de artículos científicos en PDF a Markdown

Este notebook toma artículos científicos en formato **PDF** y los convierte a **Markdown** (`.md`),
conservando en la medida de lo posible la estructura del documento: títulos, secciones, listas y tablas.

El caso de uso principal es procesar artículos científicos (por ejemplo, papers sobre lluvia,
movimientos en masa, sistemas de alerta temprana, radar meteorológico, etc.) para poder:

- Leerlos y anotarlos más fácilmente en texto plano / Markdown.
- Indexarlos o resumirlos con herramientas de IA (LLMs) que trabajan mejor con texto plano.
- Incluir fragmentos convertidos en reportes técnicos (Overleaf/LaTeX admite Markdown vía `pandoc`).

**Librería principal:** [`pymupdf4llm`](https://pymupdf.readthedocs.io/en/latest/pymupdf4llm/index.html),
construida sobre `PyMuPDF` (`fitz`). Es rápida, no requiere GPU ni servicios externos, y está pensada
específicamente para generar Markdown "amigable para LLMs" a partir de PDFs, con soporte razonable
para tablas y para el layout de dos columnas típico de los artículos científicos.

> Nota: si un PDF es un **escaneo** (imágenes de páginas, sin texto real embebido), este método no
> extraerá texto útil. Para esos casos se necesita OCR — ver la última sección del notebook.


## 1. Instalación de dependencias

Solo es necesario ejecutar esta celda una vez (o cuando cambies de entorno).


In [ ]:
# Instalación de dependencias
# pymupdf4llm ya incluye PyMuPDF (fitz) como dependencia
%pip install -q pymupdf4llm tqdm


## 2. Importar librerías


In [ ]:
import re
from pathlib import Path

import pymupdf4llm
from tqdm.auto import tqdm


## 3. Configuración de rutas

- `INPUT_DIR`: carpeta donde están los PDF de los artículos científicos.
- `OUTPUT_DIR`: carpeta donde se guardarán los `.md` generados.
- `IMAGES_DIR`: carpeta donde se guardarán las imágenes/figuras extraídas de cada PDF (opcional).

Ajusta estas rutas según la estructura de tu proyecto.


In [ ]:
INPUT_DIR = Path("pdfs")            # carpeta con los PDF de entrada
OUTPUT_DIR = Path("markdown")       # carpeta donde se guardarán los .md
IMAGES_DIR = Path("markdown/imagenes")  # carpeta para las figuras extraídas

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
print(f"Se encontraron {len(pdf_files)} archivo(s) PDF en '{INPUT_DIR}':")
for f in pdf_files:
    print(" -", f.name)


## 4. Función para convertir un PDF a Markdown

`pymupdf4llm.to_markdown` hace la mayor parte del trabajo: detecta encabezados por tamaño de fuente,
listas, tablas y el orden de lectura en documentos multi-columna (común en artículos científicos).

Parámetros útiles:

- `write_images=True`: extrae figuras/imágenes de las páginas y las guarda como archivos, insertando
  la referencia `![](ruta)` en el Markdown resultante.
- `page_chunks=False`: si se pone en `True`, devuelve una lista de diccionarios (uno por página) en
  vez de un solo string; útil si luego quieres procesar el documento página por página.


In [ ]:
def pdf_to_markdown(pdf_path: Path, output_dir: Path, images_dir: Path | None = None) -> Path:
    """Convierte un único PDF a un archivo Markdown.

    Parameters
    ----------
    pdf_path : Path
        Ruta al archivo PDF de entrada.
    output_dir : Path
        Carpeta donde se guardará el archivo .md resultante.
    images_dir : Path | None
        Si se indica, se extraen las imágenes del PDF hacia esa carpeta.

    Returns
    -------
    Path
        Ruta del archivo .md generado.
    """
    extra_kwargs = {}
    if images_dir is not None:
        images_dir.mkdir(parents=True, exist_ok=True)
        extra_kwargs.update(
            write_images=True,
            image_path=str(images_dir),
            image_format="png",
            dpi=150,
        )

    md_text = pymupdf4llm.to_markdown(str(pdf_path), **extra_kwargs)

    output_path = output_dir / (pdf_path.stem + ".md")
    output_path.write_text(md_text, encoding="utf-8")
    return output_path


## 5. Limpieza opcional del Markdown

Los artículos científicos suelen traer "ruido" heredado del PDF: encabezados/pies de página repetidos
en cada hoja, palabras cortadas con guion al final de línea (por el layout de dos columnas) y saltos
de línea de más. Esta función aplica limpiezas simples y seguras; ajústala según lo que veas en tus
propios documentos.


In [ ]:
def limpiar_markdown(texto: str) -> str:
    """Aplica limpiezas simples a un texto Markdown extraído de un PDF científico."""

    # 1) Unir palabras cortadas por guion al final de línea, ej: "modelo-\nmiento" -> "modelomiento"
    texto = re.sub(r"(\w)-\n(\w)", r"\1\2", texto)

    # 2) Colapsar 3+ líneas en blanco seguidas a solo 2 (un párrafo de separación)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    # 3) Quitar espacios en blanco al final de cada línea
    texto = re.sub(r"[ \t]+\n", "\n", texto)

    return texto.strip() + "\n"


## 6. Procesamiento por lotes (batch)

Convierte todos los PDF encontrados en `INPUT_DIR`, aplica la limpieza y guarda el resultado en
`OUTPUT_DIR`. Si un archivo falla (por ejemplo, un PDF corrupto o protegido), se reporta el error y
se continúa con los siguientes.


In [ ]:
resultados = []

for pdf_path in tqdm(pdf_files, desc="Convirtiendo PDFs"):
    try:
        # Subcarpeta de imágenes por artículo, para no mezclar figuras de distintos PDFs
        img_dir = IMAGES_DIR / pdf_path.stem

        salida = pdf_to_markdown(pdf_path, OUTPUT_DIR, images_dir=img_dir)

        texto_limpio = limpiar_markdown(salida.read_text(encoding="utf-8"))
        salida.write_text(texto_limpio, encoding="utf-8")

        resultados.append((pdf_path.name, str(salida), "OK"))
    except Exception as e:
        resultados.append((pdf_path.name, None, f"ERROR: {e}"))

print("\nResumen:")
for nombre, salida, estado in resultados:
    print(f" - {nombre}: {estado}")


## 7. Vista previa de un resultado

Muestra las primeras líneas del Markdown generado para el primer PDF procesado exitosamente,
renderizado como Markdown dentro del notebook.


In [ ]:
from IPython.display import Markdown, display

exitosos = [r for r in resultados if r[2] == "OK"]

if exitosos:
    _, ruta_md, _ = exitosos[0]
    contenido = Path(ruta_md).read_text(encoding="utf-8")
    print(f"Vista previa de: {ruta_md}\n")
    display(Markdown(contenido[:3000]))
else:
    print("Todavía no hay archivos convertidos. Revisa INPUT_DIR y vuelve a ejecutar.")


## 8. Convertir un único PDF puntual

Celda de conveniencia para convertir un solo artículo sin correr todo el lote, por ejemplo cuando
acabas de descargar un paper nuevo.


In [ ]:
# Ejemplo de uso:
# ruta_pdf = INPUT_DIR / "nombre_del_articulo.pdf"
# salida = pdf_to_markdown(ruta_pdf, OUTPUT_DIR, images_dir=IMAGES_DIR / ruta_pdf.stem)
# Path(salida).write_text(limpiar_markdown(salida.read_text(encoding="utf-8")), encoding="utf-8")
# print("Guardado en:", salida)


## 9. Notas, límites y alternativas

- **PDFs escaneados (sin texto real):** si `pymupdf4llm` devuelve texto vacío o basura, el PDF
  probablemente es una imagen escaneada. En ese caso se necesita OCR antes de convertir a Markdown,
  por ejemplo con [`ocrmypdf`](https://ocrmypdf.readthedocs.io/) (`pip install ocrmypdf`, requiere
  Tesseract instalado en el sistema) para generar primero un PDF con capa de texto, y luego correr
  este mismo notebook sobre ese PDF ya "OCRizado".

- **Tablas y ecuaciones complejas:** `pymupdf4llm` detecta tablas simples razonablemente bien, pero
  ecuaciones matemáticas (frecuentes en papers de hidrología/geotecnia) normalmente se pierden o se
  convierten en texto plano poco legible. Si necesitas mejor fidelidad en ecuaciones y tablas
  complejas, vale la pena probar como alternativa
  [`docling`](https://github.com/docling-project/docling) (`pip install docling`) o
  [`marker-pdf`](https://github.com/VikParuchuri/marker) (`pip install marker-pdf`), ambos más
  pesados (usan modelos de layout) pero más precisos en documentos científicos complejos.

- **Múltiples columnas:** `pymupdf4llm` ya intenta detectar el orden de lectura en layouts de dos
  columnas, pero conviene revisar manualmente el `.md` de salida en artículos con figuras/tablas
  incrustadas entre columnas, ya que el orden puede alterarse.

- **Uso posterior en LaTeX/Overleaf:** puedes convertir estos `.md` a `.tex` con `pandoc`:
  `pandoc articulo.md -o articulo.tex`.
